# 📘 Databricks Cloud → VNet Data Gateway Migration

This notebook migrates Azure Databricks data sources in Power BI from **Personal Cloud connections** and **Shareable Cloud connections** to a **VNet Data Gateway** using Semantic Link (SemPy).

### What it does automatically:
1. Discovers the VNet Gateway ID (Fabric API)
2. Reads connection details from each dataset's current datasource
3. Classifies connections as **Personal Cloud** or **Shareable Cloud**
4. Creates VNet gateway datasources using the naming convention:
   - **Personal**: `{ConnType}_Pers_{DatabricksServerName}_{User}`
   - **Shareable**: `{ConnType}_Shareable_{DatabricksServerName}_{connID}`
5. Reuses an existing VNet connection if one for the same ConnType, Server, User and personal/shareable type already exists
6. Binds each dataset to the VNet gateway using the appropriate connection

### Prerequisites:
- VNet Data Gateway already created (datasources can be empty)
- Fabric notebook with appropriate permissions
- `semantic-link` pre-installed (default in Fabric Spark 3.4+)

### ⚠️ After running:
Go to **Power BI Service → Settings → Manage connections and gateways** to configure credentials on newly created gateway datasources, then test a refresh.

---
## Step 1: Setup & Configuration

In [ ]:
# %pip install semantic-link  # uncomment if not pre-installed

import sempy.fabric as fabric
import pandas as pd
import json
import requests as http_requests
import uuid

from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import *

pbi_client = fabric.FabricRestClient()
PBI_BASE   = "v1.0/myorg"

print("✅ Semantic Link initialized.\n")

In [ ]:
# ======================================================================
# 👉 CUSTOMER: Fill in the values below
# ======================================================================

WORKSPACE_ID = "270ae2f5-c8f6-4dd2-8c7a-75d051bf67a8"

# Power BI reports Databricks under these datasourceType values
DATABRICKS_TYPES = ["Databricks", "AzureDatabricks", "Extension"]

# Credential type for Databricks on the VNet gateway
CREDENTIAL_TYPE = "OAuth2"

# Optional: filter datasets by name (set to None to process all)
FILTER_BY_NAME = None

print("📝 Configuration")
print(f"   Workspace:        {WORKSPACE_ID}")
print(f"   DS Types:         {DATABRICKS_TYPES}")
print(f"   Credential Type:  {CREDENTIAL_TYPE}")
print(f"   Name Filter:      {FILTER_BY_NAME or '(all datasets)'}")

---
## Step 0.5: Log Table Setup

Creates (or reuses) a Delta table in the default Lakehouse to log migration activity for this run.


In [ ]:
# Step 0.5: Migration log table setup

RUN_ID = str(uuid.uuid4())
WORKSPACE_NAME = ""

MIGRATION_LOG_TABLE = "migration_log"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {MIGRATION_LOG_TABLE} (
  run_id STRING,
  run_timestamp TIMESTAMP,
  workspace_id STRING,
  workspace_name STRING,
  dataset_id STRING,
  dataset_name STRING,
  dataset_owner_email STRING,
  datasource_type STRING,
  datasource_host STRING,
  datasource_http_path STRING,
  datasource_connection_id STRING,
  vnet_datasource_name STRING,
  vnet_datasource_id STRING,
  has_non_databricks_sources BOOLEAN,
  non_databricks_source_types STRING,
  takeover_attempted BOOLEAN,
  takeover_success BOOLEAN,
  takeover_error STRING,
  migration_status STRING,
  migration_error STRING
) USING DELTA
""")

migration_log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("run_timestamp", TimestampType(), True),
    StructField("workspace_id", StringType(), True),
    StructField("workspace_name", StringType(), True),
    StructField("dataset_id", StringType(), True),
    StructField("dataset_name", StringType(), True),
    StructField("dataset_owner_email", StringType(), True),
    StructField("datasource_type", StringType(), True),
    StructField("datasource_host", StringType(), True),
    StructField("datasource_http_path", StringType(), True),
    StructField("datasource_connection_id", StringType(), True),
    StructField("vnet_datasource_name", StringType(), True),
    StructField("vnet_datasource_id", StringType(), True),
    StructField("has_non_databricks_sources", BooleanType(), True),
    StructField("non_databricks_source_types", StringType(), True),
    StructField("takeover_attempted", BooleanType(), True),
    StructField("takeover_success", BooleanType(), True),
    StructField("takeover_error", StringType(), True),
    StructField("migration_status", StringType(), True),
    StructField("migration_error", StringType(), True),
])

def log_migration_entry(entry: dict):
    payload = {k: entry.get(k) for k in migration_log_schema.fieldNames()}
    payload["run_id"] = payload.get("run_id") or RUN_ID
    payload["workspace_id"] = payload.get("workspace_id") or WORKSPACE_ID
    payload["workspace_name"] = payload.get("workspace_name") or WORKSPACE_NAME

    df_log = spark.createDataFrame([Row(**payload)], schema=migration_log_schema)
    df_log = df_log.withColumn("run_timestamp", current_timestamp())
    df_log.write.format("delta").mode("append").saveAsTable(MIGRATION_LOG_TABLE)

print(f"🧾 Migration log table ready: {MIGRATION_LOG_TABLE}")
print(f"🆔 Run ID: {RUN_ID}")


---
## Step 2: Discover VNet Gateway ID

Uses the **Fabric API** with `mssparkutils` token.  
Reference: [List Gateways](https://learn.microsoft.com/en-us/rest/api/fabric/core/gateways/list-gateways)

In [ ]:
KNOWN_VNET_GATEWAY_IDS = set()
VNET_GATEWAY_ID = None

try:
    fabric_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com/")
    fabric_headers = {
        "Authorization": f"Bearer {fabric_token}",
        "Content-Type":  "application/json",
    }

    gw_response = http_requests.get(
        "https://api.fabric.microsoft.com/v1/gateways",
        headers=fabric_headers,
    )
    gw_response.raise_for_status()
    gateways = gw_response.json().get("value", [])

    if gateways:
        df_gw = pd.json_normalize(gateways)
        display_cols = [c for c in ["id", "displayName", "type", "capacityId"] if c in df_gw.columns]
        print(f"Found {len(gateways)} gateway(s):\n")
        display(df_gw[display_cols])

        for gw in gateways:
            gw_type = gw.get("type", "").lower()
            if gw_type in ("virtualnetwork", "vnet"):
                KNOWN_VNET_GATEWAY_IDS.add(gw["id"])

        if KNOWN_VNET_GATEWAY_IDS:
            VNET_GATEWAY_ID = list(KNOWN_VNET_GATEWAY_IDS)[0]
            for gw in gateways:
                if gw["id"] == VNET_GATEWAY_ID:
                    print(f"\n🎯 VNet gateway: {gw.get('displayName', 'N/A')} ({VNET_GATEWAY_ID})")
                    break
        else:
            print("\n⚠️  Gateways found but none are VNet type.")
    else:
        print("⚠️  Fabric API returned 0 gateways.")

except Exception as e:
    print(f"❌ Fabric Gateway API failed: {e}")

if not VNET_GATEWAY_ID:
    VNET_GATEWAY_ID = "<SET_MANUALLY>"

print(f"\nVNET_GATEWAY_ID = {VNET_GATEWAY_ID}")

if VNET_GATEWAY_ID == "<SET_MANUALLY>":
    print("\n❗ Set VNET_GATEWAY_ID manually in the override cell below.")

In [ ]:
# ======================================================================
# 👉 OVERRIDE (Optional): Uncomment if auto-discovery failed
# ======================================================================

# VNET_GATEWAY_ID = "<YOUR_VNET_GATEWAY_GUID>"
# KNOWN_VNET_GATEWAY_IDS = {VNET_GATEWAY_ID}

In [ ]:
# Resolve workspace display name for logging
WORKSPACE_NAME = WORKSPACE_ID
try:
    ws_resp = pbi_client.get(f"{PBI_BASE}/groups/{WORKSPACE_ID}")
    ws_resp.raise_for_status()
    WORKSPACE_NAME = ws_resp.json().get("name", WORKSPACE_ID)
except Exception as e:
    print(f"⚠️  Could not resolve workspace name: {e}")

print(f"Workspace name resolved: {WORKSPACE_NAME}")


---
## Step 3: Scan Datasets, Get Owners & Raw Connection Details

Scans all datasets, identifies Databricks datasources, and classifies each as:
- **Personal Cloud** – no gateway, owned by a specific user
- **Shareable Cloud** – shared cloud connection with a `connectionId`
- **✅ VNet Gateway** – already migrated

The full raw API response is captured for each datasource.

In [ ]:
# Helper functions

def is_databricks_source(src: dict) -> bool:
    ds_type = src.get("datasourceType", "")
    if ds_type in ("Databricks", "AzureDatabricks"):
        return True
    if ds_type == "Extension":
        conn_str = json.dumps(src.get("connectionDetails", {})).lower()
        return "databricks" in conn_str or "azuredatabricks.net" in conn_str
    return False


def is_shareable_connection(src: dict) -> bool:
    """Returns True when the datasource uses a shareable cloud connection.
    Shareable connections carry a non-empty 'connectionId' field and are
    not bound to a known VNet gateway.
    """
    conn_id = src.get("connectionId", "")
    gw_id   = src.get("gatewayId", "")
    return bool(conn_id) and gw_id not in KNOWN_VNET_GATEWAY_IDS


def classify_connection(src: dict) -> str:
    gw_id   = src.get("gatewayId", "")
    conn_id = src.get("connectionId", "")
    if gw_id in KNOWN_VNET_GATEWAY_IDS:
        return "✅ VNet Gateway"
    if conn_id:
        return "🔗 Shareable Cloud"
    return "☁️ Personal Cloud"


def needs_migration(src: dict) -> bool:
    gw_id = src.get("gatewayId", "")
    return gw_id not in KNOWN_VNET_GATEWAY_IDS


def get_username(email: str) -> str:
    if not email:
        return "unknown"
    return email.split("@")[0].replace(".", "_").replace(" ", "_").lower()


def get_server_name(host: str) -> str:
    """Extract a short, filesystem-safe server identifier from a Databricks host.

    'adb-123456789012345.7.azuredatabricks.net' → 'adb-123456789012345'
    """
    if not host:
        return "unknown"
    host = host.replace("https://", "").replace("http://", "").strip("/")
    return host.split(".")[0] or "unknown"


def build_vnet_connection_name(
    ds_type: str,
    server_name: str,
    user: str,
    is_shareable: bool,
    conn_id: str = "",
) -> str:
    """Build the VNet datasource display name following the agreed conventions.

    Personal  : {ConnType}_Pers_{DatabricksServerName}_{User}
    Shareable : {ConnType}_Shareable_{DatabricksServerName}_{connID[:8]}
    """
    if is_shareable:
        short_id = conn_id[:8] if conn_id else "shared"
        return f"{ds_type}_Shareable_{server_name}_{short_id}"
    return f"{ds_type}_Pers_{server_name}_{user}"


def make_conn_map_key(
    ds_type: str,
    server_name: str,
    user: str,
    is_shareable: bool,
    conn_id: str = "",
) -> str:
    """Build a deduplication key for the connection map.

    Personal  : 'ConnType|server|user|personal'
    Shareable : 'ConnType|server|connID[:8]|shareable'
    """
    if is_shareable:
        short_id = conn_id[:8] if conn_id else "shared"
        return f"{ds_type}|{server_name}|{short_id}|shareable"
    return f"{ds_type}|{server_name}|{user}|personal"


def extract_host_and_path(src: dict) -> tuple:
    """
    Extracts the raw host and httpPath from a datasource,
    handling all possible nesting levels.
    Returns (host_string, httpPath_string) — both are plain strings.
    """
    conn = src.get("connectionDetails", {})

    raw_host = conn.get("host", conn.get("server", conn.get("path", "")))
    raw_http = conn.get("httpPath", conn.get("database", conn.get("HTTPPath", "")))

    # If host is already a JSON string like '{"host":"adb-xxx","httpPath":"/sql/..."}'
    # then parse it out
    if isinstance(raw_host, str) and raw_host.strip().startswith("{"):
        try:
            parsed = json.loads(raw_host)
            if isinstance(parsed, dict):
                raw_host = parsed.get("host", parsed.get("server", raw_host))
                if not raw_http:
                    raw_http = parsed.get("httpPath", parsed.get("database", ""))
        except (json.JSONDecodeError, TypeError):
            pass

    # Clean host: remove https:// and trailing slashes
    if isinstance(raw_host, str):
        raw_host = raw_host.replace("https://", "").replace("http://", "").strip("/")

    return (raw_host, raw_http)


def build_create_connection_details(src: dict) -> str:
    """
    Builds connectionDetails matching the EXACT working format:

    "{
        \\"extensionDataSourceKind\\": \\"Databricks\\",
        \\"extensionDataSourcePath\\": \\"{\\\\\\"host\\\\\\":\\\\\\"adb-xxx\\\\\\",\\\\\\"httpPath\\\\\\":\\\\\\"/sql/...\\\\\\"}\\"}
    }"

    extensionDataSourcePath is a STRINGIFIED JSON of {host, httpPath}.
    The whole connectionDetails is also stringified.
    """
    ds_type = src.get("datasourceType", "")
    host, http_path = extract_host_and_path(src)

    if ds_type == "Extension":
        path_obj = {}
        if host:
            path_obj["host"] = host
        if http_path:
            path_obj["httpPath"] = http_path

        conn_obj = {
            "extensionDataSourceKind": "Databricks",
            "extensionDataSourcePath": json.dumps(path_obj),
        }
    else:
        conn_obj = {"server": host}
        if http_path:
            conn_obj["httpPath"] = http_path

    return json.dumps(conn_obj)


def get_non_databricks_types(sources: list) -> str:
    non_types = sorted({(s.get("datasourceType") or "Unknown") for s in sources if not is_databricks_source(s)})
    return ",".join(non_types)

print("✅ Helper functions defined.")


In [ ]:
# Scan all datasets and get owners + raw datasource details

response = pbi_client.get(f"{PBI_BASE}/groups/{WORKSPACE_ID}/datasets")
response.raise_for_status()
all_datasets = response.json().get("value", [])

datasets = all_datasets
if FILTER_BY_NAME:
    datasets = [ds for ds in datasets if FILTER_BY_NAME.lower() in ds["name"].lower()]

print(f"Scanning {len(datasets)} dataset(s)...\n")

inventory = []

for ds in datasets:
    ds_id   = ds["id"]
    ds_name = ds["name"]
    ds_owner_email = ds.get("configuredBy", "")
    ds_owner_user  = get_username(ds_owner_email)

    try:
        resp = pbi_client.get(f"{PBI_BASE}/groups/{WORKSPACE_ID}/datasets/{ds_id}/datasources")
        resp.raise_for_status()

        sources = resp.json().get("value", [])
        all_ds_types = sorted({(s.get("datasourceType") or "Unknown") for s in sources})
        non_databricks_source_types = get_non_databricks_types(sources)
        has_non_databricks_sources = bool(non_databricks_source_types)

        for src in sources:
            if is_databricks_source(src):
                conn          = src.get("connectionDetails", {})
                conn_type     = classify_connection(src)
                shareable     = is_shareable_connection(src)
                src_conn_id   = src.get("connectionId", "")
                host_raw      = conn.get("server", conn.get("host", conn.get("path", "")))
                host_clean, _ = extract_host_and_path(src)
                srv_name      = get_server_name(host_clean)
                ds_type_val   = src.get("datasourceType", "")
                conn_key      = make_conn_map_key(
                    ds_type_val, srv_name, ds_owner_user, shareable, src_conn_id
                )

                inventory.append({
                    "dataset_id":      ds_id,
                    "dataset_name":    ds_name,
                    "owner_email":     ds_owner_email,
                    "owner_user":      ds_owner_user,
                    "datasourceType":  ds_type_val,
                    "host":            host_raw,
                    "host_clean":      host_clean,
                    "server_name":     srv_name,
                    "httpPath":        conn.get("httpPath", conn.get("database", conn.get("HTTPPath", ""))),
                    "gatewayId":       src.get("gatewayId", ""),
                    "connectionId":    src_conn_id,
                    "is_shareable":    shareable,
                    "connection_type": conn_type,
                    "needs_migration": needs_migration(src),
                    "conn_map_key":    conn_key,
                    "all_datasource_types": ",".join(all_ds_types),
                    "has_non_databricks_sources": has_non_databricks_sources,
                    "non_databricks_source_types": non_databricks_source_types,
                    "raw_src":         src,
                })
    except Exception as e:
        print(f"⚠️  Could not read '{ds_name}': {e}")

df_all = pd.DataFrame(inventory)

if df_all.empty:
    print("❌ No Databricks datasources found.")
else:
    to_migrate   = df_all["needs_migration"].sum()
    already_ok   = len(df_all) - to_migrate
    personal_cnt = (~df_all["is_shareable"] & df_all["needs_migration"]).sum()
    share_cnt    = (df_all["is_shareable"]  & df_all["needs_migration"]).sum()
    mixed_cnt    = df_all[df_all["has_non_databricks_sources"]].dataset_id.nunique()
    print(f"Databricks datasources found            : {len(df_all)}")
    print(f"☁️  To migrate – Personal Cloud          : {personal_cnt}")
    print(f"🔗 To migrate – Shareable Cloud          : {share_cnt}")
    print(f"✅ Already on VNet Gateway               : {already_ok}")
    print(f"⚠️  Datasets with non-Databricks sources : {mixed_cnt}\n")
    display(df_all[[
        "dataset_name", "owner_email", "owner_user",
        "datasourceType", "host", "httpPath",
        "is_shareable", "connectionId", "connection_type",
        "has_non_databricks_sources", "non_databricks_source_types",
    ]])


---
## Step 4: Create Datasources on VNet Gateway

Creates **one VNet datasource per unique combination** of (ConnType, DatabricksServer, User/connID, personal/shareable) using the naming conventions:

| Source type | Naming convention |
|-------------|-------------------|
| Personal cloud | `{ConnType}_Pers_{DatabricksServerName}_{User}` |
| Shareable cloud | `{ConnType}_Shareable_{DatabricksServerName}_{connID[:8]}` |

If a VNet connection with the same name already exists it is **reused** — no duplicate is created.

For **Extension** type the `connectionDetails` uses:
```json
{"extensionDataSourceKind": "Databricks", "extensionDataSourcePath": "adb-xxx..."}
```

For **Databricks** / **AzureDatabricks** native type:
```json
{"server": "adb-xxx...", "httpPath": "/sql/..."}
```

Reference: [Create Datasource API](https://learn.microsoft.com/en-us/rest/api/power-bi/gateways/create-datasource)


In [ ]:
# Step 4: Create Datasources on VNet Gateway
#
# One VNet datasource per unique (ConnType, Server, User|connID, personal|shareable).
# Naming:
#   Personal  → {ConnType}_Pers_{Server}_{User}
#   Shareable → {ConnType}_Shareable_{Server}_{connID[:8]}

# connection_map: conn_map_key → VNet datasource/connection ID
connection_map = {}

if df_all.empty or VNET_GATEWAY_ID.startswith("<"):
    print("Nothing to create (no datasources found or gateway not set).")
else:
    df_to_create = df_all[df_all["needs_migration"] == True].copy()

    if df_to_create.empty:
        print("✅ All datasources already on VNet Gateway.")
    else:
        # ── Refresh Fabric token ──
        fabric_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com/")
        fabric_headers = {
            "Authorization": f"Bearer {fabric_token}",
            "Content-Type":  "application/json",
        }

        # ── List existing gateway datasources ──
        existing_by_name = {}
        try:
            existing_resp = pbi_client.get(f"{PBI_BASE}/gateways/{VNET_GATEWAY_ID}/datasources")
            if existing_resp.status_code == 200:
                existing_gw_datasources = existing_resp.json().get("value", [])
                print(f"📋 Existing datasources on VNet gateway : {len(existing_gw_datasources)}")
                for eds in existing_gw_datasources:
                    for name_key in ["datasourceName", "name"]:
                        ds_name_gw = eds.get(name_key, "").lower()
                        if ds_name_gw:
                            existing_by_name[ds_name_gw] = eds.get("id", "")
        except Exception:
            pass

        # ── List existing Fabric connections ──
        existing_fabric_conns = {}
        try:
            fc_resp = http_requests.get(
                "https://api.fabric.microsoft.com/v1/connections",
                headers=fabric_headers,
            )
            if fc_resp.status_code == 200:
                for c in fc_resp.json().get("value", []):
                    c_name = c.get("displayName", "").lower()
                    if c_name:
                        existing_fabric_conns[c_name] = c.get("id", "")
        except Exception:
            pass

        # ── Build unique connections to create ──
        # Key: conn_map_key  →  {raw_src, owner_email, owner_user, is_shareable, conn_id}
        conns_to_create = {}
        for _, row in df_to_create.iterrows():
            key = row["conn_map_key"]
            if key not in conns_to_create:
                conns_to_create[key] = {
                    "raw_src":      row["raw_src"],
                    "owner_email":  row["owner_email"],
                    "owner_user":   row["owner_user"],
                    "datasourceType": row["datasourceType"],
                    "server_name":  row["server_name"],
                    "host_clean":   row["host_clean"],
                    "is_shareable": row["is_shareable"],
                    "conn_id":      row["connectionId"],
                }

        p_count = sum(1 for v in conns_to_create.values() if not v["is_shareable"])
        s_count = sum(1 for v in conns_to_create.values() if v["is_shareable"])
        print(f"\n🔑 Unique connections to provision: {len(conns_to_create)} "
              f"(☁️ {p_count} personal + 🔗 {s_count} shareable)\n")

        # ── Create or reuse ──
        for key, info in conns_to_create.items():
            src          = info["raw_src"]
            ds_type      = info["datasourceType"]
            server_name  = info["server_name"]
            user         = info["owner_user"]
            is_shareable = info["is_shareable"]
            conn_id      = info["conn_id"]

            target_name = build_vnet_connection_name(
                ds_type, server_name, user, is_shareable, conn_id
            )
            kind_label = "🔗 Shareable" if is_shareable else "☁️ Personal"

            # Check legacy gateway datasources
            if target_name.lower() in existing_by_name:
                ds_id_found = existing_by_name[target_name.lower()]
                connection_map[key] = ds_id_found
                print(f"   ♻️  [{kind_label}] '{target_name}' already on gateway (ID: {ds_id_found})")
                continue

            # Check Fabric connections
            if target_name.lower() in existing_fabric_conns:
                ds_id_found = existing_fabric_conns[target_name.lower()]
                connection_map[key] = ds_id_found
                print(f"   ♻️  [{kind_label}] '{target_name}' already as Fabric connection (ID: {ds_id_found})")
                continue

            # ── Build path object ──
            host, http_path = extract_host_and_path(src)
            path_obj = {}
            if host:
                path_obj["host"] = host
            if http_path:
                path_obj["httpPath"] = http_path

            # ── Create body ──
            create_body = {
                "gatewayId":         VNET_GATEWAY_ID,
                "displayName":       target_name,
                "connectivityType":  "VirtualNetworkGateway",
                "connectionDetails": {
                    "creationMethod": "Databricks.Catalogs",
                    "path":           json.dumps(path_obj),
                    "type":           "Databricks",
                },
                "privacyLevel":      "Organizational",
                "credentialDetails": {
                    "singleSignOnType":     "None",
                    "connectionEncryption": "NotEncrypted",
                    "skipTestConnection":   True,
                    "credentials": {
                        "credentialType": "Key",
                        "username":       "admin",
                        "key":            "skip-test-placeholder",
                    },
                },
            }

            print(f"   📤 [{kind_label}] Creating '{target_name}'...")
            print(f"      host    : {host}")
            print(f"      httpPath: {http_path}")

            try:
                create_resp = http_requests.post(
                    "https://api.fabric.microsoft.com/v1/connections",
                    headers=fabric_headers,
                    json=create_body,
                )
                create_resp.raise_for_status()
                new_conn    = create_resp.json()
                new_conn_id = new_conn.get("id", "")
                connection_map[key] = new_conn_id
                print(f"      ✅ Created (ID: {new_conn_id})\n")

            except Exception as e:
                err_detail = ""
                try:
                    err_detail = create_resp.text
                except Exception:
                    pass
                print(f"      ❌ Failed: {e}")
                if err_detail:
                    print(f"      Response: {err_detail}")
                print(f"      Body sent: {json.dumps(create_body, indent=2)}\n")

        # ── Summary ──
        print(f"\n📋 Connection map ({len(connection_map)} entries):")
        for k, v in connection_map.items():
            kind = "🔗 Shareable" if "shareable" in k else "☁️ Personal"
            print(f"   {kind}  {k}  →  {v}")

        if connection_map:
            print("\n" + "=" * 70)
            print("✅ Datasources created with skipTestConnection=true.")
            print("   Each owner must configure their real credentials:")
            print("   Power BI → ⚙️ → Manage connections and gateways")
            print("   → Click connection → Edit credentials → Sign in with OAuth2")
            print("=" * 70)


---
## Step 5: Review Candidates for Rebind

**Review this list before proceeding to Step 6.**

In [ ]:
if not df_all.empty:
    df_migrate = df_all[df_all["needs_migration"] == True].copy()
else:
    df_migrate = pd.DataFrame()

if df_migrate.empty:
    print("✅ All Databricks datasources are already on the VNet Gateway!")
    datasets_to_update = pd.DataFrame()
else:
    df_migrate["target_conn_id"] = df_migrate["conn_map_key"].map(connection_map)
    df_migrate["target_ds_name"] = df_migrate.apply(
        lambda r: build_vnet_connection_name(
            r["datasourceType"],
            r["server_name"],
            r["owner_user"],
            r["is_shareable"],
            r["connectionId"],
        ),
        axis=1,
    )

    datasets_to_update = df_migrate.drop_duplicates(subset=["dataset_id"])
    datasets_for_bind = datasets_to_update[datasets_to_update["has_non_databricks_sources"] != True]

    print(f"🎯 {len(datasets_for_bind)} dataset(s) will be bound:\n")
    display(datasets_for_bind[[
        "dataset_name", "owner_email", "datasourceType",
        "host", "httpPath", "is_shareable", "target_ds_name", "target_conn_id",
    ]])
    print(f"\nTarget VNet Gateway: {VNET_GATEWAY_ID}")

    mixed_sources = datasets_to_update[datasets_to_update["has_non_databricks_sources"] == True]
    if not mixed_sources.empty:
        print("\n⚠️  Mixed-source datasets (will NOT be bound):")
        for _, r in mixed_sources.iterrows():
            print(f"   - {r['dataset_name']}: {r['non_databricks_source_types']}")

    missing = datasets_for_bind[datasets_for_bind["target_conn_id"].isna()]["conn_map_key"].unique()
    if len(missing) > 0:
        print(f"\n⚠️  Keys without a gateway datasource (will be skipped): {list(missing)}")

    print("\n⚠️  Review above. Run the next cell to apply.")


---
## Step 6: Bind Datasets to VNet Gateway

Reference: [BindToGateway API](https://learn.microsoft.com/en-us/rest/api/power-bi/datasets/bind-to-gateway-in-group)

In [ ]:
results = []

if df_migrate.empty:
    print("Nothing to process.")
elif VNET_GATEWAY_ID.startswith("<"):
    print("❌ VNET_GATEWAY_ID not set. Go back to Step 2.")
elif not connection_map:
    print("❌ No datasources on VNet gateway. Check Step 4.")
else:
    for _, row in datasets_to_update.iterrows():
        ds_id       = row["dataset_id"]
        ds_name     = row["dataset_name"]
        conn_key    = row["conn_map_key"]
        target_name = row["target_ds_name"]

        target_ds_id = connection_map.get(conn_key)

        log_entry = {
            "run_id": RUN_ID,
            "workspace_id": WORKSPACE_ID,
            "workspace_name": WORKSPACE_NAME,
            "dataset_id": ds_id,
            "dataset_name": ds_name,
            "dataset_owner_email": row.get("owner_email", ""),
            "datasource_type": row.get("datasourceType", ""),
            "datasource_host": row.get("host_clean", row.get("host", "")),
            "datasource_http_path": row.get("httpPath", ""),
            "datasource_connection_id": row.get("connectionId", ""),
            "vnet_datasource_name": target_name,
            "vnet_datasource_id": target_ds_id or "",
            "has_non_databricks_sources": bool(row.get("has_non_databricks_sources", False)),
            "non_databricks_source_types": row.get("non_databricks_source_types", ""),
            "takeover_attempted": False,
            "takeover_success": False,
            "takeover_error": "",
            "migration_status": "",
            "migration_error": "",
        }

        if bool(row.get("has_non_databricks_sources", False)):
            warn_msg = (
                f"⚠️  '{ds_name}' skipped — contains non-Databricks sources: "
                f"{row.get('non_databricks_source_types', '')}. Manual migration required."
            )
            print(warn_msg)
            log_entry["migration_status"] = "skipped_mixed_sources"
            log_migration_entry(log_entry)
            results.append({
                "dataset": ds_name,
                "conn_key": conn_key,
                "target_name": target_name,
                "status": "skipped_mixed_sources",
                "reason": row.get("non_databricks_source_types", ""),
            })
            continue

        if not target_ds_id:
            print(f"⏭️  '{ds_name}' — no VNet datasource for key '{conn_key}', skipping.")
            log_entry["migration_status"] = "skipped_no_vnet_ds"
            log_entry["migration_error"] = f"no datasource for {conn_key}"
            log_migration_entry(log_entry)
            results.append({
                "dataset": ds_name,
                "conn_key": conn_key,
                "target_name": target_name,
                "status": "skipped",
                "reason": f"no datasource for {conn_key}",
            })
            continue

        takeover_ok = False
        takeover_error = ""
        log_entry["takeover_attempted"] = True

        try:
            takeover_resp = pbi_client.post(
                f"{PBI_BASE}/groups/{WORKSPACE_ID}/datasets/{ds_id}/Default.TakeOver"
            )
            if takeover_resp.status_code == 200:
                takeover_ok = True
            else:
                takeover_error = f"HTTP {takeover_resp.status_code}: {takeover_resp.text}"
                print(f"❌ TakeOver failed for '{ds_name}': {takeover_error}")
        except Exception as e:
            takeover_error = str(e)
            print(f"❌ TakeOver failed for '{ds_name}': {takeover_error}")

        log_entry["takeover_success"] = takeover_ok
        log_entry["takeover_error"] = takeover_error

        try:
            bind_body = {
                "gatewayObjectId":     VNET_GATEWAY_ID,
                "datasourceObjectIds": [target_ds_id],
            }

            bind_resp = pbi_client.post(
                f"{PBI_BASE}/groups/{WORKSPACE_ID}/datasets/{ds_id}/Default.BindToGateway",
                json=bind_body,
            )
            bind_resp.raise_for_status()

            print(f"✅ '{ds_name}' → bound to '{target_name}'")
            log_entry["migration_status"] = "migrated"
            log_migration_entry(log_entry)
            results.append({
                "dataset": ds_name,
                "conn_key": conn_key,
                "target_name": target_name,
                "status": "success",
                "reason": "",
            })

        except Exception as e:
            print(f"❌ '{ds_name}' → FAILED: {e}")
            log_entry["migration_status"] = "failed"
            log_entry["migration_error"] = str(e)
            log_migration_entry(log_entry)
            results.append({
                "dataset": ds_name,
                "conn_key": conn_key,
                "target_name": target_name,
                "status": "failed",
                "reason": str(e),
            })

mixed_not_migrated = [r for r in results if r.get("status") == "skipped_mixed_sources"]
print("\n⚠️ Datasets NOT Migrated (mixed sources)")
if mixed_not_migrated:
    for r in mixed_not_migrated:
        print(f"   - {r['dataset']}: {r.get('reason', '')}")
else:
    print("   None")


---
## Step 7: Summary & Verification

In [ ]:
from pyspark.sql import functions as F

df_results = pd.DataFrame(results)

if not df_results.empty:
    s = (df_results["status"] == "success").sum()
    k = (df_results["status"] == "skipped").sum()
    m = (df_results["status"] == "skipped_mixed_sources").sum()
    f = (df_results["status"] == "failed").sum()
    print(f"\n✅ Bound    : {s}")
    print(f"⏭️  Skipped  : {k}")
    print(f"⚠️  Mixed    : {m}")
    print(f"❌ Failed   : {f}\n")
    display(df_results)

    success_rows = df_results[df_results["status"] == "success"]
    if not success_rows.empty:
        check_name = success_rows.iloc[0]["dataset"]
        check_id   = datasets_to_update[
            datasets_to_update["dataset_name"] == check_name
        ].iloc[0]["dataset_id"]

        resp = pbi_client.get(f"{PBI_BASE}/groups/{WORKSPACE_ID}/datasets/{check_id}/datasources")
        resp.raise_for_status()

        print(f"\n🔎 Spot-check: '{check_name}'\n")
        for src in resp.json().get("value", []):
            conn_type = classify_connection(src)
            conn = src.get("connectionDetails", {})
            print(f"   Type:       {src.get('datasourceType')}  |  {conn_type}")
            print(f"   Connection: {json.dumps(conn)}\n")
else:
    print("No datasets were processed.")

print("\n⚠️ Datasets NOT Migrated (from migration_log)")
df_not_migrated = spark.table(MIGRATION_LOG_TABLE).filter(
    (F.col("run_id") == RUN_ID) &
    (F.col("migration_status").isin("skipped_mixed_sources", "failed"))
)

display(df_not_migrated)
not_migrated_count = df_not_migrated.count()
print(f"⚠️ {not_migrated_count} dataset(s) were NOT migrated this run.")

print("=" * 70)
print("✅ COMPLETE")
print("")
print("   Next steps:")
print("   1. Go to Power BI Service → Settings → Manage connections")
print("   2. For each new datasource, configure the owner's credentials")
print("   3. Test a refresh to verify connectivity")
print("")
print("   Datasources created (configure credentials for each):")
for key, ds_id in connection_map.items():
    kind = "🔗 Shareable" if "shareable" in key else "☁️ Personal"
    print(f"      {kind}  {key}  →  {ds_id}")
print("=" * 70)


In [ ]:
# ======================================================================
# 📋 PLAN B: List ALL Databricks datasets using personal cloud connections
#            across the entire tenant
#
# Output: DataFrame with workspace, dataset, datasource, owner details
#         + export to CSV for sharing with dataset owners
# ======================================================================

import json
import time
import pandas as pd
import requests as http_requests

# ── Tokens ──
try:
    pbi_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
except Exception:
    pbi_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com/")

headers = {
    "Authorization": f"Bearer {pbi_token}",
    "Content-Type":  "application/json",
}

PBI_API = "https://api.powerbi.com/v1.0/myorg"

# ======================================================================
# Step 1: Get ALL workspaces (admin endpoint)
# ======================================================================
print("🔍 Step 1: Fetching all workspaces from tenant...")
all_workspaces = []
top = 5000
skip = 0

while True:
    ws_resp = http_requests.get(
        f"{PBI_API}/admin/groups?$top={top}&$skip={skip}&$filter=type eq 'Workspace'",
        headers=headers,
    )
    if ws_resp.status_code != 200:
        print(f"   ⚠️  Status {ws_resp.status_code}: {ws_resp.text[:500]}")
        break
    batch = ws_resp.json().get("value", [])
    if not batch:
        break
    all_workspaces.extend(batch)
    print(f"   Fetched {len(all_workspaces)} workspaces...")
    if len(batch) < top:
        break
    skip += top

print(f"✅ Total workspaces: {len(all_workspaces)}\n")

# ======================================================================
# Step 2: For each workspace, get datasets + their datasources
# ======================================================================
print("🔍 Step 2: Scanning datasets and datasources across all workspaces...")
print("   (This may take a few minutes for large tenants)\n")

results = []
errors = []
ws_count = len(all_workspaces)

for idx, ws in enumerate(all_workspaces):
    ws_id   = ws.get("id", "")
    ws_name = ws.get("name", "")

    if (idx + 1) % 50 == 0 or idx == 0:
        print(f"   Processing workspace {idx + 1}/{ws_count}: {ws_name}")

    # Get datasets in this workspace (admin endpoint)
    try:
        ds_resp = http_requests.get(
            f"{PBI_API}/admin/groups/{ws_id}/datasets",
            headers=headers,
        )
        if ds_resp.status_code == 429:
            retry_after = int(ds_resp.headers.get("Retry-After", 30))
            print(f"   ⏳ Rate limited. Sleeping {retry_after}s...")
            time.sleep(retry_after)
            ds_resp = http_requests.get(
                f"{PBI_API}/admin/groups/{ws_id}/datasets",
                headers=headers,
            )

        if ds_resp.status_code != 200:
            continue

        datasets = ds_resp.json().get("value", [])
    except Exception as e:
        errors.append({"workspace_id": ws_id, "error": str(e)})
        continue

    for dataset in datasets:
        dataset_id   = dataset.get("id", "")
        dataset_name = dataset.get("name", "")
        configured_by       = dataset.get("configuredBy", "")
        configured_by_email = dataset.get("configuredByEmail", configured_by)

        # Get datasources for this dataset
        try:
            src_resp = http_requests.get(
                f"{PBI_API}/admin/datasets/{dataset_id}/datasources",
                headers=headers,
            )
            if src_resp.status_code == 429:
                retry_after = int(src_resp.headers.get("Retry-After", 30))
                print(f"   ⏳ Rate limited. Sleeping {retry_after}s...")
                time.sleep(retry_after)
                src_resp = http_requests.get(
                    f"{PBI_API}/admin/datasets/{dataset_id}/datasources",
                    headers=headers,
                )

            if src_resp.status_code != 200:
                continue

            datasources = src_resp.json().get("value", [])
        except Exception as e:
            errors.append({"dataset_id": dataset_id, "error": str(e)})
            continue

        for src in datasources:
            ds_type   = src.get("datasourceType", "")
            conn_str  = json.dumps(src.get("connectionDetails", {}))
            gw_id     = src.get("gatewayId", "")
            src_id    = src.get("datasourceId", "")
            src_name  = src.get("datasourceName", src.get("name", ""))
            cred_type = src.get("credentialType", src.get("credentialDetails", {}).get("credentialType", ""))

            # Filter: only Databricks-related sources
            is_dbx = False
            if ds_type in ("Databricks", "AzureDatabricks"):
                is_dbx = True
            elif ds_type == "Extension":
                if "databricks" in conn_str.lower() or "azuredatabricks.net" in conn_str.lower():
                    is_dbx = True

            if not is_dbx:
                continue

            # Classify gateway
            if not gw_id:
                gw_class = "☁️ No Gateway (Direct Cloud)"
            elif gw_id in KNOWN_VNET_GATEWAY_IDS:
                gw_class = "✅ VNet Gateway"
            else:
                gw_class = "⚠️ Personal Cloud Gateway"

            # Extract host and httpPath
            try:
                conn_details = src.get("connectionDetails", {})
                host_val = conn_details.get("host", conn_details.get("server", conn_details.get("path", "")))
                http_path_val = conn_details.get("httpPath", conn_details.get("database", ""))

                # Try to parse if host is stringified JSON
                if isinstance(host_val, str) and host_val.strip().startswith("{"):
                    try:
                        parsed = json.loads(host_val)
                        host_val = parsed.get("host", parsed.get("server", host_val))
                        if not http_path_val:
                            http_path_val = parsed.get("httpPath", "")
                    except Exception:
                        pass
            except Exception:
                host_val = ""
                http_path_val = ""

            results.append({
                "workspace_id":       ws_id,
                "workspace_name":     ws_name,
                "dataset_id":         dataset_id,
                "dataset_name":       dataset_name,
                "datasource_id":      src_id,
                "datasource_name":    src_name,
                "datasource_type":    ds_type,
                "gateway_id":         gw_id,
                "gateway_class":      gw_class,
                "credential_type":    cred_type,
                "host":               host_val,
                "http_path":          http_path_val,
                "connection_details": conn_str,
                "owner_name":         configured_by,
                "owner_email":        configured_by_email,
            })

    # Small delay to avoid rate limits
    if (idx + 1) % 20 == 0:
        time.sleep(0.5)

print(f"\n✅ Scan complete.")
print(f"   Total Databricks datasources found: {len(results)}")
if errors:
    print(f"   ⚠️  Errors encountered: {len(errors)}")

# ======================================================================
# Step 3: Build DataFrame
# ======================================================================
df_planb = pd.DataFrame(results)

if not df_planb.empty:
    # Sort: personal gateways first (the ones needing action)
    df_planb = df_planb.sort_values(
        by=["gateway_class", "owner_email", "workspace_name", "dataset_name"],
        ascending=[False, True, True, True],
    ).reset_index(drop=True)

    print(f"\n📊 Summary by gateway type:")
    print(df_planb.groupby("gateway_class").size().to_string())

    print(f"\n📊 Summary by owner:")
    print(
        df_planb[df_planb["gateway_class"] != "✅ VNet Gateway"]
        .groupby(["owner_email", "gateway_class"])
        .size()
        .to_string()
    )

    # ── Filter to only personal/cloud connections (not yet on VNet) ──
    df_personal = df_planb[df_planb["gateway_class"] != "✅ VNet Gateway"].copy()
    print(f"\n⚠️  Datasources NOT on VNet gateway: {len(df_personal)}")
    print(f"   Unique owners to contact: {df_personal['owner_email'].nunique()}")
else:
    df_personal = pd.DataFrame()
    print("\n✅ No Databricks datasources found in the tenant.")

# ======================================================================
# Step 4: Display and export
# ======================================================================
if not df_personal.empty:
    display_cols = [
        "workspace_name", "dataset_name", "datasource_name",
        "host", "http_path", "credential_type", "gateway_class",
        "owner_name", "owner_email",
    ]
    existing_cols = [c for c in display_cols if c in df_personal.columns]

    print("\n" + "=" * 70)
    print("📋 PLAN B: Datasets needing manual datasource migration")
    print("=" * 70)
    display(df_personal[existing_cols])

    # Export to CSV
    csv_path = "/lakehouse/default/Files/planb_databricks_personal_connections.csv"
    try:
        df_personal.to_csv(csv_path, index=False)
        print(f"\n💾 Exported to: {csv_path}")
    except Exception:
        # Fallback path
        csv_path = "/tmp/planb_databricks_personal_connections.csv"
        df_personal.to_csv(csv_path, index=False)
        print(f"\n💾 Exported to: {csv_path}")

    # ── Generate email template per owner ──
    print("\n" + "=" * 70)
    print("📧 EMAIL TEMPLATES (one per owner):")
    print("=" * 70)
    for owner_email, grp in df_personal.groupby("owner_email"):
        datasets_list = "\n".join([
            f"   • {row['dataset_name']} (workspace: {row['workspace_name']})"
            for _, row in grp.iterrows()
        ])
        print(f"""
To: {owner_email}
Subject: Action Required – Migrate Power BI Databricks connection to VNet Gateway

Hi {grp.iloc[0]['owner_name']},

The following Power BI dataset(s) you own are using a personal Databricks
connection that needs to be migrated to our VNet Data Gateway:

{datasets_list}

Please follow these steps:
1. Go to Power BI → ⚙️ Settings → Manage connections and gateways
2. Select the VNet gateway
3. Add a new datasource → Type: Databricks
4. Enter the host and HTTP path, then sign in with OAuth2
5. Rebind your dataset(s) to the new gateway datasource

If you need help, contact the BI team.

Thanks!
{"─" * 60}""")

    # Also export full report (including VNet ones for reference)
    if not df_planb.empty:
        csv_full = "/lakehouse/default/Files/all_databricks_connections_report.csv"
        try:
            df_planb.to_csv(csv_full, index=False)
            print(f"\n💾 Full report (all Databricks connections): {csv_full}")
        except Exception:
            csv_full = "/tmp/all_databricks_connections_report.csv"
            df_planb.to_csv(csv_full, index=False)
            print(f"\n💾 Full report: {csv_full}")
else:
    print("\n✅ All Databricks connections are already on VNet gateway. No action needed!")